# 01 — Profile Language (`eunomia.profiles/v1`) & `alexandriac` Compiler

This interactive notebook explores **Part II** of the Alexandria v0.3.0 specification:
1. Authoring an **`OdrlProfile`** and a **`SpaceProfile`** in `eunomia.profiles/v1` YAML.
2. Validating across **levels N1–N5** (`alexandriac validate`) and inspecting line-accurate diagnostics.
3. Compiling deterministic semantic artefacts (`alexandriac compile`): SHACL `shapes.ttl`, JSON Schema `schema.json`, JSON-LD `context.jsonld`, RDF `vocab.ttl`, and `space/space-lock.json`.
4. Enforcing **SemVer evolution invariants (N4)** with `-previous` (`0.1.0` → `0.2.0` deprecation → `1.0.0` retirement).
5. Publishing a multi-document `eunomia.profiles/v1` YAML stream directly to `alexandria-server` configured with `"compiler": "dsl"`.

In [ ]:
import json
import subprocess
import tempfile
from pathlib import Path
from lab import AlexandriaLab, ensure_binaries, show_artifacts, show_yaml

bin_dir = ensure_binaries()
alexandriac = bin_dir / "alexandriac"
work = Path(tempfile.mkdtemp(prefix="alexandriac-nb01-"))
print(f"Using compiler binary: {alexandriac}")

## 1. Authoring an `OdrlProfile` and `SpaceProfile`

An `OdrlProfile` declares namespace prefixes, adopted ODRL left operands, domain terms (`action`, `left_operand`, `right_operand`) with typed operator signatures, and closed/open action sets. A `SpaceProfile` binds one or more profiles into slots (`odrl`, `dcat`, etc.) and maps artifact roles (`shacl`, `json-schema`) to vhub gates (`policy_intake`, `dsp_message`).

In [ ]:
colors_v010 = '''dsl: eunomia.profiles/v1
kind: OdrlProfile

profile:
  id: https://example.com/odrl/profile/colors
  version: 0.1.0
  title:
    en: Color Labels ODRL Profile
    es: Perfil ODRL de etiquetas de color
  description:
    en: ODRL profile for applying colour labels to assets in a data space.
  publisher: GING-DIT, Universidad Politécnica de Madrid
  license: https://creativecommons.org/licenses/by/4.0/
  base_url: https://example.com/odrl/profile/colors

namespaces:
  col: https://example.com/ns/colors#

adopted:
  left_operands:
    - term: odrl:dateTime
      signature:
        datatype: xsd:dateTime
        operators: [eq, lt, lteq, gt, gteq]

terms:
  - id: putColor
    kind: action
    included_in: odrl:use
    label: { en: Put color }
    definition: { en: To assign a colour label to the Asset. }

  - id: color
    kind: left_operand
    label: { en: Color }
    definition: { en: The colour label assigned by the action. }
    signature:
      value_kind: iri
      values: [col:red, col:green, col:blue, col:yellow]
      operators: [eq, neq]
    enforcement:
      cedar_path: context.color
      pdp_level: PDP-4
      eval_timing: once

  - id: red
    kind: right_operand
    label: { en: Red }
    definition: { en: The colour red (#FF0000). }
  - id: green
    kind: right_operand
    label: { en: Green }
    definition: { en: The colour green (#00FF00). }
  - id: blue
    kind: right_operand
    label: { en: Blue }
    definition: { en: The colour blue (#0000FF). }
  - id: yellow
    kind: right_operand
    label: { en: Yellow }
    definition: { en: The colour yellow (#FFFF00). }

rules:
  actions: closed
'''

space_v100 = '''dsl: eunomia.profiles/v1
kind: SpaceProfile
space:
  id: https://example.com/spaces/demo/profile
  version: 1.0.0
  title: { en: Demo Data Space }
  operator: https://example.com/party/demo-authority
  base_url: https://example.com/spaces/demo
bindings:
  - slot: odrl
    profile: https://example.com/odrl/profile/colors
    source: internal
    version: "^0.1"
    obligation: mandatory
    conformance:
      - { role: shacl, applies_to: policy_intake }
      - { role: json-schema, applies_to: dsp_message }
'''

profile_path = work / "colors.profile.yaml"
space_path = work / "demo.space.yaml"
profile_path.write_text(colors_v010, encoding="utf-8")
space_path.write_text(space_v100, encoding="utf-8")

show_yaml("colors.profile.yaml", colors_v010, subtitle="OdrlProfile v0.1.0")
show_yaml("demo.space.yaml", space_v100, subtitle="SpaceProfile v1.0.0")

## 2. Validation (`N1`–`N5`) and Deterministic Compilation

Let's validate both YAML files with `alexandriac validate` and then compile them twice into separate directories (`out_a` and `out_b`) to verify **byte-for-byte determinism** across every generated artefact and `space/space-lock.json`.

In [ ]:
res_val = subprocess.run(
    [str(alexandriac), "validate", str(profile_path), str(space_path)],
    capture_output=True, text=True, check=True,
)
print("alexandriac validate ->", res_val.stdout.strip())

out_a = work / "out_a"
out_b = work / "out_b"
subprocess.run([str(alexandriac), "compile", "-o", str(out_a), str(profile_path), str(space_path)], check=True, capture_output=True)
subprocess.run([str(alexandriac), "compile", "-o", str(out_b), str(profile_path), str(space_path)], check=True, capture_output=True)

files_a = sorted(p.relative_to(out_a).as_posix() for p in out_a.rglob("*") if p.is_file())
for rel in files_a:
    assert (out_a / rel).read_bytes() == (out_b / rel).read_bytes(), f"Non-deterministic output in {rel}"

print(f"Verified byte-for-byte determinism across {len(files_a)} artefacts!")
show_artifacts("Compiled Bundle (OdrlProfile 0.1.0 + SpaceProfile 1.0.0)", out_a)

## 3. Compiler Diagnostic Levels (`N1` Structural & `N2` Semantic Checks)

`alexandriac` enforces strict static guarantees before any artefact can reach a data space:
- **N1**: Rejects YAML anchors/aliases/custom tags and unknown schema fields with exact line numbers.
- **N2**: Rejects undeclared prefixes, duplicate term IDs, missing enforcement signatures, and ordered comparison operators (`lt`, `gt`, ...) on unordered IRI enums.

In [ ]:
bad_anchor = work / "bad_anchor.yaml"
bad_anchor.write_text("dsl: &anchor eunomia.profiles/v1\nkind: OdrlProfile\n", encoding="utf-8")

bad_operator = work / "bad_operator.yaml"
bad_operator.write_text(colors_v010.replace("operators: [eq, neq]", "operators: [eq, lt]"), encoding="utf-8")

for label, candidate in [("N1 YAML Anchor Check", bad_anchor), ("N2 Ordered Operator on IRI Enum", bad_operator)]:
    proc = subprocess.run([str(alexandriac), "validate", str(candidate)], capture_output=True, text=True)
    assert proc.returncode == 1
    print(f"=== {label} (exit={proc.returncode}) ===")
    print(proc.stderr.strip())
    print()

## 4. SemVer Evolution Checks (`N4`) with `-previous`

Under **N4**, a term published in `0.1.0` cannot disappear abruptly or jump straight to `retired` without first being marked `status: deprecated` in an intermediate release, and retiring a term requires a **MAJOR** version bump (`1.0.0`).

In [ ]:
# 1. Invalid evolution: retiring 'yellow' in 0.2.0 (MINOR) without prior deprecation.
bad_v020 = work / "colors_bad_020.yaml"
bad_v020.write_text(
    colors_v010.replace("version: 0.1.0", "version: 0.2.0").replace(
        "  - id: yellow\n    kind: right_operand",
        "  - id: yellow\n    kind: right_operand\n    status: retired",
    ),
    encoding="utf-8",
)
proc_bad = subprocess.run(
    [str(alexandriac), "validate", "-previous", str(profile_path), str(bad_v020)],
    capture_output=True, text=True,
)
assert proc_bad.returncode == 1
print("=== Rejected Direct Retirement in 0.2.0 ===")
print(proc_bad.stderr.strip())

# 2. Valid 2-step lifecycle: 0.1.0 -> 0.2.0 (deprecated) -> 1.0.0 (retired in MAJOR).
v020_depr = work / "colors_020.yaml"
v020_depr.write_text(
    colors_v010.replace("version: 0.1.0", "version: 0.2.0").replace(
        "  - id: yellow\n    kind: right_operand",
        "  - id: yellow\n    kind: right_operand\n    status: deprecated\n    deprecated_since: 0.2.0",
    ),
    encoding="utf-8",
)
res_020 = subprocess.run(
    [str(alexandriac), "validate", "-previous", str(profile_path), str(v020_depr)],
    capture_output=True, text=True, check=True,
)
print("\n0.1.0 -> 0.2.0 (deprecated):", res_020.stdout.strip())

v100_ret = work / "colors_100.yaml"
v100_ret.write_text(
    colors_v010.replace("version: 0.1.0", "version: 1.0.0")
    .replace("values: [col:red, col:green, col:blue, col:yellow]", "values: [col:red, col:green, col:blue]")
    .replace(
        "  - id: yellow\n    kind: right_operand",
        "  - id: yellow\n    kind: right_operand\n    status: retired\n    deprecated_since: 0.2.0\n    retired_since: 1.0.0",
    ),
    encoding="utf-8",
)
res_100 = subprocess.run(
    [str(alexandriac), "validate", "-previous", str(v020_depr), str(v100_ret)],
    capture_output=True, text=True, check=True,
)
print("0.2.0 -> 1.0.0 (retired in MAJOR):", res_100.stdout.strip())

## 5. End-to-End DSL Compilation Inside `alexandria-server` (`compiler: "dsl"`)

`alexandria-server` embeds the exact same `compiler_dsl` engine! Let's launch an isolated Hub configured with `compiler="dsl"`, submit a multi-document YAML stream (`OdrlProfile` + `SpaceProfile`), validate it via `:validate`, propose, and approve `1.0.0` into a signed W3C VC 2.0 `VersionRecord`.

In [ ]:
lab = AlexandriaLab(namespace="dsl-demo", compiler="dsl", initial_clock="2026-09-01T10:00:00Z")
try:
    lab.start_hub_stack()
    ana = lab.token("ana", admin=True)
    sergio = lab.token("sergio")

    lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "dsl-demo"})
    lab.request("POST", f"{lab.hub_url}/v1/namespaces/dsl-demo/stewards", token=ana, body={"iss": lab.idp_url, "sub": "sergio"})

    yaml_bundle = colors_v010.strip() + "\n---\n" + space_v100.strip() + "\n"
    st, hdr, draft = lab.request(
        "POST", f"{lab.hub_url}/v1/namespaces/dsl-demo/drafts",
        token=sergio, headers={"Content-Type": "application/yaml"}, body=yaml_bundle,
    )
    assert st == 201
    draft_id = draft["id"].rsplit("/", 1)[-1]

    st, _, val_rep = lab.request("POST", f"{lab.hub_url}/v1/namespaces/dsl-demo/drafts/{draft_id}:validate", token=sergio)
    assert st == 200 and val_rep["valid"] is True

    st, _, cand = lab.request("POST", f"{lab.hub_url}/v1/namespaces/dsl-demo/drafts/{draft_id}:propose", token=sergio)
    assert st == 201
    cand_id = cand["id"].rsplit("/", 1)[-1]

    st, _, rec_vc = lab.request("POST", f"{lab.hub_url}/v1/namespaces/dsl-demo/candidates/{cand_id}:approve", token=sergio)
    assert st == 201
    assert "VersionRecord" in rec_vc["type"]
    assert rec_vc["proof"]["cryptosuite"] == "eddsa-jcs-2022"
finally:
    lab.close()